In [7]:
# !python -m venv .venv && source .venv/bin/activate
# ! pip install -r requirements.txt
# !pip install pyarrow

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 100)

In [2]:
# reuse the pattern from module 3
def read_station_csv(station_id: str) -> pd.DataFrame:
    """
    Read NOAA GHCN-D CSV data for a given station_id.
    """
    path = f"s3://noaa-ghcn-pds/csv/by_station/{station_id}.csv"
    df = pd.read_csv(
        path,storage_options={"anon": True},
        dtype={"Q_FLAG": "object", "M_FLAG": "object", "S_FLAG": "object"},
        parse_dates=["DATE"]
    )

    return df

def make_wide_from_csv(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot long-format into wide format with one row per day and columns as ELEMENT values. 
    Convert TMIN/TMAX from 0.1C to C.
    """
    df_wide = df.pivot(index="DATE", columns="ELEMENT", values="DATA_VALUE")
    if "TMIN" in df_wide.columns:
        df_wide["TMIN"] = df_wide["TMIN"] / 10.0
    if "TMAX" in df_wide.columns:
        df_wide["TMAX"] = df_wide["TMAX"] / 10.0
    return df_wide



In [3]:
station_id = "USC00087205"  # Plant City, Florida

df_raw = read_station_csv(station_id)
display(df_raw.head())

df_wide = make_wide_from_csv(df_raw)
df_wide = df_wide.sort_index()

display(df_wide.head())
print("Available variables:", df_wide.columns.tolist())

,ID,DATE,ELEMENT,DATA_VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
0,USC00087205,1892-09-01,TMAX,322,NaN,NaN,6,NaN
1,USC00087205,1892-09-02,TMAX,317,NaN,NaN,6,NaN
2,USC00087205,1892-09-03,TMAX,317,NaN,NaN,6,NaN
3,USC00087205,1892-09-04,TMAX,322,NaN,NaN,6,NaN
4,USC00087205,1892-09-05,TMAX,333,NaN,NaN,6,NaN


ELEMENT,DAPR,MDPR,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WT01,WT03,WT04,WT06,WT08,WT11,WT14,WT16
DATE,,,,,,,,,,,,,,,,
1892-09-01,NaN,NaN,NaN,NaN,NaN,32.2,20.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1892-09-02,NaN,NaN,NaN,NaN,NaN,31.7,20.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1892-09-03,NaN,NaN,NaN,NaN,NaN,31.7,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1892-09-04,NaN,NaN,NaN,NaN,NaN,32.2,21.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1892-09-05,NaN,NaN,NaN,NaN,NaN,33.3,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Available variables: ['DAPR', 'MDPR', 'PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN', 'TOBS', 'WT01', 'WT03', 'WT04', 'WT06', 'WT08', 'WT11', 'WT14', 'WT16']


In [4]:
# Check TMIN exist?
if "TMIN" not in df_wide.columns:
    raise ValueError(f"Station {station_id} does not contain TMIN.")

# Choose to 1991–2020
tmin = df_wide[["TMIN"]].copy()
tmin = tmin.loc["1991-01-01":"2020-12-31"].copy()

# Add time components
tmin["year"] = tmin.index.year
tmin["month"] = tmin.index.month
tmin["day"] = tmin.index.day

# Convert to F
tmin["tmin_f"] = tmin["TMIN"] * 9.0 / 5.0 + 32.0

display(tmin.head())

ELEMENT,TMIN,year,month,day,tmin_f
DATE,,,,,
1991-01-01,16.7,1991,1,1,62.06
1991-01-02,18.3,1991,1,2,64.94
1991-01-03,17.8,1991,1,3,64.04
1991-01-04,17.8,1991,1,4,64.04
1991-01-05,17.2,1991,1,5,62.96


In [5]:
# Select October, November, December, January
season_months = [10, 11, 12, 1]
tmin_season = tmin[tmin["month"].isin(season_months)].copy()

display(tmin_season.head())

ELEMENT,TMIN,year,month,day,tmin_f
DATE,,,,,
1991-01-01,16.7,1991,1,1,62.06
1991-01-02,18.3,1991,1,2,64.94
1991-01-03,17.8,1991,1,3,64.04
1991-01-04,17.8,1991,1,4,64.04
1991-01-05,17.2,1991,1,5,62.96


In [6]:
tmin_season["freeze_32"] = (tmin_season["tmin_f"] <= 32.0).astype(int)
tmin_season["freeze_28"] = (tmin_season["tmin_f"] <= 28.0).astype(int)

display(tmin_season[["TMIN", "tmin_f", "freeze_32", "freeze_28"]].head(10))

ELEMENT,TMIN,tmin_f,freeze_32,freeze_28
DATE,,,,
1991-01-01,16.7,62.06,0,0
1991-01-02,18.3,64.94,0,0
1991-01-03,17.8,64.04,0,0
1991-01-04,17.8,64.04,0,0
1991-01-05,17.2,62.96,0,0
1991-01-06,19.4,66.92,0,0
1991-01-07,20.6,69.08,0,0
1991-01-08,17.8,64.04,0,0
1991-01-09,15.0,59.00,0,0


In [7]:
monthly_counts = (
    tmin_season
    .groupby(["year", "month"])
    .agg(
        days_32=("freeze_32", "sum"),
        days_28=("freeze_28", "sum"),
        total_days=("tmin_f", "size"),
    )
    .reset_index()
)

print("Monthly frost or freeze counts (first few rows):")
display(monthly_counts.head(12))

Monthly frost or freeze counts (first few rows):


,year,month,days_32,days_28,total_days
0,1991,1,0,0,31
1,1991,10,0,0,31
2,1991,11,0,0,30
3,1991,12,0,0,31
4,1992,1,3,0,31
5,1992,10,0,0,31
6,1992,11,0,0,30
7,1992,12,0,0,31
8,1993,1,0,0,31
9,1993,10,0,0,31


In [8]:
monthly_risk = (
    monthly_counts
    .groupby("month")[['days_32', 'days_28']]
    .mean()
)

month_name_map = {10: "Oct", 11: "Nov", 12: "Dec", 1: "Jan"}
risk_table = monthly_risk.copy()
risk_table.index = risk_table.index.map(month_name_map)

risk_table = risk_table.rename(
    columns={
        'days_32': 'Mean days Tmin ≤ 32°F',
        'days_28': 'Mean days Tmin ≤ 28°F',
    }
)

print("Mean frost and freeze risk per month (1991–2020, Plant City, FL):")
display(risk_table)

Mean frost and freeze risk per month (1991–2020, Plant City, FL):


,Mean days Tmin ≤ 32°F,Mean days Tmin ≤ 28°F
month,,
Jan,1.866667,0.500000
Oct,0.000000,0.000000
Nov,0.033333,0.000000
Dec,0.600000,0.166667


In [9]:
sstoi_path = Path("sstoi.indices")  # adjust if needed

sst = pd.read_csv(sstoi_path, delim_whitespace=True)

print("Raw SSTOI data (first few rows):")
display(sst.head())
print("Columns:", sst.columns.tolist())

Raw SSTOI data (first few rows):


/tmp/ipykernel_678/3468302648.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  sst = pd.read_csv(sstoi_path, delim_whitespace=True)


,YR,MON,NINO1+2,ANOM,NINO3,ANOM.1,NINO4,ANOM.2,NINO3.4,ANOM.3
0,1982,1,24.28,-0.24,25.84,0.17,28.01,-0.21,26.65,0.08
1,1982,2,25.38,-0.72,26.26,-0.11,27.99,-0.11,26.54,-0.20
2,1982,3,25.22,-1.38,26.92,-0.25,28.18,-0.05,27.09,-0.14
3,1982,4,24.57,-1.16,27.52,-0.05,28.61,0.10,27.83,0.02
4,1982,5,24.00,-0.62,27.70,0.49,29.19,0.40,28.37,0.49


Columns: ['YR', 'MON', 'NINO1+2', 'ANOM', 'NINO3', 'ANOM.1', 'NINO4', 'ANOM.2', 'NINO3.4', 'ANOM.3']


In [10]:
rename_map = {
    'YR': 'year',
    'MON': 'month',
    'NINO1+2': 'nino12',
    'ANOM': 'nino12_anom',
    'NINO3': 'nino3',
    'ANOM.1': 'nino3_anom',
    'NINO4': 'nino4',
    'ANOM.2': 'nino4_anom',
    'NINO3.4': 'nino34',
    'ANOM.3': 'nino34_anom',
}

sst = sst.rename(columns=rename_map)
print("After renaming:")
display(sst.head())

After renaming:


,year,month,nino12,nino12_anom,nino3,nino3_anom,nino4,nino4_anom,nino34,nino34_anom
0,1982,1,24.28,-0.24,25.84,0.17,28.01,-0.21,26.65,0.08
1,1982,2,25.38,-0.72,26.26,-0.11,27.99,-0.11,26.54,-0.20
2,1982,3,25.22,-1.38,26.92,-0.25,28.18,-0.05,27.09,-0.14
3,1982,4,24.57,-1.16,27.52,-0.05,28.61,0.10,27.83,0.02
4,1982,5,24.00,-0.62,27.70,0.49,29.19,0.40,28.37,0.49


In [11]:
sst_sub = sst.query('1991 <= year <= 2020').copy()
sst_sub = sst_sub[sst_sub['month'].isin([10, 11, 12, 1])].copy()

print('ENSO subset years:', sst_sub['year'].min(), sst_sub['year'].max())
print('ENSO subset months:', sorted(sst_sub['month'].unique()))
display(sst_sub.head())

ENSO subset years: 1991 2020
ENSO subset months: [np.int64(1), np.int64(10), np.int64(11), np.int64(12)]


,year,month,nino12,nino12_anom,nino3,nino3_anom,nino4,nino4_anom,nino34,nino34_anom
108,1991,1,23.73,-0.78,25.63,-0.05,28.62,0.40,26.89,0.33
117,1991,10,21.09,0.22,25.36,0.27,29.31,0.63,27.41,0.64
118,1991,11,21.92,0.29,25.93,0.73,29.12,0.44,27.71,0.89
119,1991,12,23.13,0.29,26.30,1.03,29.21,0.77,28.14,1.50
120,1992,1,24.59,0.07,26.86,1.18,28.80,0.58,28.23,1.67


In [12]:
merged = pd.merge(
    monthly_counts,sst_sub,
    on=['year', 'month'],
    how='inner',
)

print('Merged DataFrame shape:', merged.shape)
display(merged.head(12))

Merged DataFrame shape: (120, 13)


,year,month,days_32,days_28,total_days,nino12,nino12_anom,nino3,nino3_anom,nino4,nino4_anom,nino34,nino34_anom
0,1991,1,0,0,31,23.73,-0.78,25.63,-0.05,28.62,0.40,26.89,0.33
1,1991,10,0,0,31,21.09,0.22,25.36,0.27,29.31,0.63,27.41,0.64
2,1991,11,0,0,30,21.92,0.29,25.93,0.73,29.12,0.44,27.71,0.89
3,1991,12,0,0,31,23.13,0.29,26.30,1.03,29.21,0.77,28.14,1.50
4,1992,1,3,0,31,24.59,0.07,26.86,1.18,28.80,0.58,28.23,1.67
5,1992,10,0,0,31,20.99,0.12,24.66,-0.42,28.45,-0.24,26.27,-0.50
6,1992,11,0,0,30,21.37,-0.26,24.97,-0.23,28.37,-0.30,26.64,-0.18
7,1992,12,0,0,31,22.41,-0.43,25.10,-0.17,28.48,0.03,26.78,0.14
8,1993,1,0,0,31,24.25,-0.27,25.66,-0.01,28.32,0.09,26.73,0.16
9,1993,10,0,0,31,21.19,0.32,25.28,0.20,28.66,-0.02,26.96,0.19


In [13]:
corr_cols = ['days_28', 'nino12_anom', 'nino3_anom', 'nino4_anom', 'nino34_anom']

corr_matrix = merged[corr_cols].corr()

print("Correlation matrix (Pearson r):")
display(corr_matrix)

target = 'days_28'
enso_indices = ['nino12_anom', 'nino3_anom', 'nino4_anom', 'nino34_anom']

abs_corr = {idx: abs(corr_matrix.loc[target, idx]) for idx in enso_indices}

print("\nAbsolute correlations with monthly freeze days (Tmin ≤ 28°F):")
for idx, val in abs_corr.items():
    print(f"{idx}: {val:.3f}")

best_index = max(abs_corr, key=abs_corr.get)
print(f"\nENSO index with highest |r| with freeze days: {best_index} (|r| = {abs_corr[best_index]:.3f})")

Correlation matrix (Pearson r):


,days_28,nino12_anom,nino3_anom,nino4_anom,nino34_anom
days_28,1.000000,-0.135629,-0.122463,-0.135148,-0.112039
nino12_anom,-0.135629,1.000000,0.870064,0.549747,0.755858
nino3_anom,-0.122463,0.870064,1.000000,0.805135,0.963471
nino4_anom,-0.135148,0.549747,0.805135,1.000000,0.909278
nino34_anom,-0.112039,0.755858,0.963471,0.909278,1.000000



Absolute correlations with monthly freeze days (Tmin ≤ 28°F):
nino12_anom: 0.136
nino3_anom: 0.122
nino4_anom: 0.135
nino34_anom: 0.112

ENSO index with highest |r| with freeze days: nino12_anom (|r| = 0.136)


In [14]:
# fig, ax = plt.subplots(figsize=(6, 4))
# ax.scatter(merged['nino34_anom'], merged['days_28'])
# ax.axhline(0, linestyle='--', alpha=0.5)
# ax.axvline(0, linestyle='--', alpha=0.5)
# ax.set_xlabel('Niño3.4 SST anomaly (°C)')
# ax.set_ylabel('Number of days Tmin ≤ 28°F')
# ax.set_title('Monthly freeze days vs Niño3.4 anomalies\nPlant City, FL (1991–2020, Oct–Jan)')
# plt.tight_layout()
# plt.show()
